# Report Visualisations

In [3]:
import sys
import pathlib
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Add src/ to path so data_loaders is importable
ROOT = pathlib.Path.cwd().parent   # notebooks/ -> project root
sys.path.insert(0, str(ROOT / "src"))

from data_loaders import load_cpih_monthly, load_cpih_fy_indices, load_lcf_shares

OUTPUT = ROOT / "data" / "output"
%matplotlib inline

In [6]:
plt.rcParams.update({
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.family":       "serif",
    "font.size":         9,
    "axes.titlesize":    11,
    "figure.dpi":        150,
})

TENURE_4 = ["social_rent", "private_rent", "own_outright", "own_mortgage"]

TENURE_LABELS = {
    "social_rent":  "Social Rent",
    "private_rent": "Private Rent",
    "own_outright": "Own Outright",
    "own_mortgage": "Own Mortgage",
}

TENURE_COLOURS = {
    "social_rent":  "#1b9e77",
    "private_rent": "#d95f02",
    "own_outright": "#7570b3",
    "own_mortgage": "#e7298a",
}

COICOP_MAP = {
    "share_01_food_non_alcoholic":  "Food",
    "share_02_alcohol_tobacco":     "Alcohol & Tobacco",
    "share_03_clothing_footwear":   "Clothing",
    "share_04_housing_fuel_power":  "Housing & Utilities",
    "share_05_furnishings":         "Furnishings",
    "share_06_health":              "Health",
    "share_07_transport":           "Transport",
    "share_08_communication":       "Communication",
    "share_09_recreation_culture":  "Recreation",
    "share_10_education":           "Education",
    "share_11_restaurants_hotels":  "Restaurants",
    "share_12_misc_goods_services": "Misc. Goods",
}

OFFICIAL_COICOP = list(COICOP_MAP.keys())

In [10]:
shares = load_lcf_shares()
monthly = load_cpih_monthly()
fy_index = load_cpih_fy_indices()

# archetype_name, archetype_value, year, inflation_rate
group_inflation = pd.read_csv(OUTPUT / "group_inflation_rates.csv")

# archetype name, archetype value, year, COICOP label, contribution
coicop_contributions = pd.read_csv(OUTPUT / "inflation_decomposition.csv")

print("These are the shapes of each database (for debugging):")
print(f"- LCF shares: {shares.shape}")
print(f"- CPIH monthly: {monthly.shape}")
print(f"- CPIH FY index: {fy_index.shape}")
print(f"- Inflation: {group_inflation.shape}")
print(f"- Decomposition: {coicop_contributions.shape}")

These are the shapes of each database (for debugging):
- LCF shares: (45500, 25)
- CPIH monthly: (458, 20)
- CPIH FY index: (9, 17)
- Inflation: (135, 4)
- Decomposition: (1890, 5)


### Figure 2: Data quality audit
Two panels:
- (a) How much data is genuinely missing per column?
- (b) How many households report 0 spending per COICOP division? This is not a data quality issue but reflects that LCF is a biweekly diary where households may have reasons for 0 spending in a category (most households don't pay for education or tobacco every 2 weeks).

In [ ]:
share_cols = [c for c in OFFICIAL_COICOP if c in shares.columns]

cols = ["gross_weekly_income", "equivalised_income", "total_expenditure", "weight"] + share_cols
available = [c for c in cols if c in shares.columns]

# For each COICOP category, % of households reported spending absolutely nothing? 
zero_percentage_categories = ((shares[share_cols] == 0).sum() / len(shares) * 100).sort_values()

labels = [COICOP_MAP.get(c,c) for c in zero_percentage_categories.index]

fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(labels, zero_percentage_categories.values, color="#3498DB")

